# Question 8: Graph Algorithms for Transport Systems

## Introduction

A transport authority models its road network as a **weighted graph**: cities are vertices, roads are weighted edges (distance in km). This notebook implements traversal (BFS/DFS), shortest-path computation (Dijkstra), and network-wide cost minimization (Minimum Spanning Tree via Kruskal's algorithm).

## (c) Graph Using Adjacency Lists

(Presented first since traversal and shortest-path algorithms below depend on it.)

In [1]:
graph = {
    "Kigali":  [("Musanze", 60), ("Huye", 135)],
    "Musanze": [("Kigali", 60), ("Rubavu", 35)],
    "Rubavu":  [("Musanze", 35)],
    "Huye":    [("Kigali", 135)],
}

# unweighted view, for plain BFS/DFS traversal
unweighted_graph = {city: [n for n, w in neighbours] for city, neighbours in graph.items()}

for city, roads in graph.items():
    print(f"{city:8} -> {roads}")

Kigali   -> [('Musanze', 60), ('Huye', 135)]
Musanze  -> [('Kigali', 60), ('Rubavu', 35)]
Rubavu   -> [('Musanze', 35)]
Huye     -> [('Kigali', 135)]


## (a) BFS and DFS Traversal

In [2]:
from collections import deque

def bfs(graph, start):
    visited = {start}
    order = []
    queue = deque([start])
    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbour in graph[node]:
            if neighbour not in visited:
                visited.add(neighbour)
                queue.append(neighbour)
    return order

def dfs(graph, node, visited=None, order=None):
    if visited is None:
        visited, order = set(), []
    visited.add(node)
    order.append(node)
    for neighbour in graph[node]:
        if neighbour not in visited:
            dfs(graph, neighbour, visited, order)
    return order

bfs_order = bfs(unweighted_graph, "Kigali")
dfs_order = dfs(unweighted_graph, "Kigali")
print("BFS from Kigali:", bfs_order)
print("DFS from Kigali:", dfs_order)
assert set(bfs_order) == set(unweighted_graph.keys())
assert set(dfs_order) == set(unweighted_graph.keys())
print("Both traversals visit every reachable city ✔  |  Time complexity: O(V + E)")

BFS from Kigali: ['Kigali', 'Musanze', 'Huye', 'Rubavu']
DFS from Kigali: ['Kigali', 'Musanze', 'Rubavu', 'Huye']
Both traversals visit every reachable city ✔  |  Time complexity: O(V + E)


## (b) Dijkstra's Algorithm

Computes shortest-path distances from a source city to every other city using a min-priority queue (binary heap).

In [3]:
import heapq

def dijkstra(graph, start):
    distances = {node: float('inf') for node in graph}
    distances[start] = 0
    pq = [(0, start)]
    while pq:
        current_distance, current_node = heapq.heappop(pq)
        if current_distance > distances[current_node]:
            continue  # stale entry, skip
        for neighbour, weight in graph[current_node]:
            distance = current_distance + weight
            if distance < distances[neighbour]:
                distances[neighbour] = distance
                heapq.heappush(pq, (distance, neighbour))
    return distances

shortest = dijkstra(graph, "Kigali")
print("Shortest distances from Kigali:")
for city, d in shortest.items():
    print(f"  {city:8}: {d} km")

# Manual verification against the known road network
assert shortest["Kigali"] == 0
assert shortest["Musanze"] == 60
assert shortest["Rubavu"] == 95     # Kigali -> Musanze -> Rubavu = 60 + 35
assert shortest["Huye"] == 135
print("\nAll shortest distances manually verified against the graph ✔")

Shortest distances from Kigali:
  Kigali  : 0 km
  Musanze : 60 km
  Rubavu  : 95 km
  Huye    : 135 km

All shortest distances manually verified against the graph ✔


## (d) Minimum Spanning Tree (Kruskal's Algorithm)

An MST connects every city using the **minimum total road length**, with no cycles. Kruskal's algorithm sorts all edges by weight and greedily adds each edge that doesn't create a cycle, using a **Disjoint Set (Union-Find)** to detect cycles in near-constant time. (Note: simply sorting edges, as in an early draft, is necessary but not sufficient — cycle detection via union-find is what makes it a correct MST algorithm.)

In [4]:
class DisjointSet:
    def __init__(self, vertices):
        self.parent = {v: v for v in vertices}
    def find(self, item):
        if self.parent[item] != item:
            self.parent[item] = self.find(self.parent[item])  # path compression
        return self.parent[item]
    def union(self, x, y):
        root_x, root_y = self.find(x), self.find(y)
        if root_x != root_y:
            self.parent[root_y] = root_x

def kruskal(vertices, edges):
    mst = []
    ds = DisjointSet(vertices)
    for weight, u, v in sorted(edges):
        if ds.find(u) != ds.find(v):
            mst.append((u, v, weight))
            ds.union(u, v)
    return mst

vertices = ["Kigali", "Musanze", "Rubavu", "Huye"]
edges = [
    (35, "Musanze", "Rubavu"),
    (60, "Kigali", "Musanze"),
    (135, "Kigali", "Huye"),
]

mst = kruskal(vertices, edges)
print("Minimum Spanning Tree edges:")
for u, v, w in mst:
    print(f"  {u} -- {v}  ({w} km)")

total_cost = sum(w for _, _, w in mst)
print(f"\nTotal MST cost: {total_cost} km")

assert len(mst) == len(vertices) - 1, "a spanning tree on n vertices must have exactly n-1 edges"
assert total_cost == 35 + 60 + 135
print("MST structurally valid (n-1 edges, no cycles) ✔")

Minimum Spanning Tree edges:
  Musanze -- Rubavu  (35 km)
  Kigali -- Musanze  (60 km)
  Kigali -- Huye  (135 km)

Total MST cost: 230 km
MST structurally valid (n-1 edges, no cycles) ✔


### Complexity Summary

| Algorithm | Time Complexity |
|---|---|
| BFS / DFS | O(V + E) |
| Dijkstra (binary heap) | O((V + E) log V) |
| Kruskal (with Union-Find) | O(E log E) |

**Conclusion:** BFS/DFS efficiently explore connectivity, Dijkstra finds shortest point-to-point routes, and Kruskal's algorithm finds the cheapest way to keep the *entire* road network connected — three complementary tools a transport authority needs for different planning questions.